In [4]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import constants

pi = constants.pi 
c = constants.c


def create_constant_ice_layer(altitudes, layer_bottom, layer_top, ice_density):
    """
    Create a constant ice layer with specified density between given altitudes.

    Parameters:
    altitudes (numpy array): Array of altitude values in meters.
    layer_bottom (float): Bottom altitude of the ice layer in meters.
    layer_top (float): Top altitude of the ice layer in meters.
    ice_density (float): Density of ice particles in particles/m^3.

    Returns:
    numpy array: Array of particle densities corresponding to the input altitudes.
    """
    n = np.zeros_like(altitudes)  # Initialize an array of zeros
    in_layer = (altitudes >= layer_bottom) & (altitudes <= layer_top)  # Find indices within the layer
    n[in_layer] = ice_density  # Set density for those indices
    return n

def sigma_scattering(frequency, volume, A):
    """
    Calculate the scattering cross section for ice crystals.
    
    #Parameters:
    frequency (float): The frequency of the incident light (in Hz).
    volume (float): The volume of the ice crystal particle (in m^3).
    A (float): A constant that depends on the properties of the ice crystal (dielectric properties, shape, polarizability).
    
    Returns:
    float: The scattering cross section (in m^2).
    """
    w = 2 * pi * frequency #angular frequency

    sigma_sca = w**4*volume**2*np.abs(A)**2/(6*pi*c**4)
    
    return sigma_sca

def sigma_absorption(frequency, volume, A):
    """
    Calculate the absorption cross section for ice crystals.
    
    #Parameters:
    frequency (float): The frequency of the incident light (in Hz).
    volume (float): The volume of the ice crystal particle (in m^3).
    A (float): A constant that depends on the properties of the ice crystal (dielectric properties, shape, polarizability).
    
    Returns:
    float: The absorption cross section (in m^2).
    """
    w = 2 * pi * frequency #angular frequency

    sigma_abs = w*volume*np.imag(A)/c

    return sigma_abs

In [5]:
#functions to choose the right version of A

def compute_depolarization_factor(aspect_ratio):
    """
    Calculates the depolarization factor (Delta) along the symmetry axis
    for a spheroid given its aspect ratio.
    
    Parameters:
    aspect_ratio (float): Ratio of symmetry axis to transverse axis (m).
                          m = 1.0 (sphere), m > 1.0 (column), m < 1.0 (plate).
    
    Returns:
    float: Depolarization factor Delta.
    """
    m = aspect_ratio
    
    if m == 1.0:
        # Sphere
        return 1.0 / 3.0
        
    elif m > 1.0:
        # Prolate spheroid (Column)
        e = np.sqrt(1.0 - (1.0 / m)**2)
        delta = ((1.0 - e**2) / e**2) * ((1.0 / (2.0 * e)) * np.log((1.0 + e) / (1.0 - e)) - 1.0)
        return delta
        
    else:
        # Oblate spheroid (Plate)
        e = np.sqrt(1.0 - m**2)
        delta = (1.0 / e**2) * (1.0 - (np.sqrt(1.0 - e**2) / e) * np.arcsin(e))
        return delta


def compute_polarizability(frequency, aspect_ratio=1.0, orientation='none'):
    """
    Computes the complex polarizability (A) or an effective A for ice crystals
    based on their aspect ratio.
    
    Parameters:
    frequency (float): Frequency in Hz.
    aspect_ratio (float): The ratio of the symmetry axis to the transverse axis (m).
    orientation (str): 'random', 'horizontal', 'vertical', or 'none' (for spherical).
    
    Returns:
    complex: An effective complex polarizability A that works with 
             sigma_scattering (using abs(A)**2) and sigma_absorption (using imag(A)).
    """
    
    # 1. Complex relative permittivity of ice at -30 deg C [cite: 213]
    eps_prime = 3.16
    eps_double_prime = 8e-3 * (frequency / 150e9) 
    eps = eps_prime + 1j * eps_double_prime
    
    # 2. Spherical case (simplest, isotropic) [cite: 212]
    if aspect_ratio == 1.0:
        return 3 * (eps - 1) / (eps + 2)
    
    # 3. Nonspherical cases: Get depolarization factor and shape [cite: 314]
    delta = compute_depolarization_factor(aspect_ratio)
    shape = 'column' if aspect_ratio > 1.0 else 'plate'

    # Calculate parallel (symmetric axis) and perpendicular polarizabilities [cite: 308, 309, 311]
    A_par = (eps - 1) / (1 + (eps - 1) * delta)
    A_perp = (eps - 1) / (1 + (eps - 1) * (1 - delta) / 2)
    
    # 4. Handle orientations
    if orientation == 'random':
        # Averaged over all orientations [cite: 315]
        abs2 = (np.abs(A_par)**2 + 2 * np.abs(A_perp)**2) / 3
        imag = (np.imag(A_par) + 2 * np.imag(A_perp)) / 3
        
    elif orientation == 'vertical':
        # Vertical polarization response [cite: 319]
        if shape == 'column':
            abs2 = np.abs(A_perp)**2
            imag = np.imag(A_perp)
        else: # plate
            abs2 = np.abs(A_par)**2
            imag = np.imag(A_par)
            
    elif orientation == 'horizontal':
        # Horizontal polarization response [cite: 322]
        if shape == 'column':
            abs2 = (np.abs(A_par)**2 + np.abs(A_perp)**2) / 2
            imag = (np.imag(A_par) + np.imag(A_perp)) / 2
        else: # plate
            abs2 = np.abs(A_perp)**2
            imag = np.imag(A_perp)
            
    else: 
        raise ValueError("Orientation must be 'random', 'horizontal', or 'vertical'.")
        
    # 5. Construct an "effective" complex A so that your existing functions work.
    # We want a complex number (x + iy) where y = imag, and x^2 + y^2 = abs2.
    real_part = np.sqrt(np.maximum(0, abs2 - imag**2))
    A_effective = real_part + 1j * imag
    
    return A_effective

def compute_gamma(frequency, aspect_ratio=0.5, process='scattering'):
    """
    Computes the gamma parameter (ratio of horizontal to vertical polarizability)
    for horizontally aligned ice crystals (Eq 9, 10, 12).
    
    Parameters:
    frequency (float): Frequency of the radiation in Hz.
    aspect_ratio (float): Ratio of symmetry axis to transverse axis (m).
    process (str): 'scattering' or 'emission'.
    
    Returns:
    float: The gamma parameter.
    """
    # 1. For spherical particles, response is isotropic so gamma = 1
    if aspect_ratio == 1.0:
        return 1.0
        
    # 2. Complex relative permittivity of ice at -30 deg C
    eps_prime = 3.16
    eps_double_prime = 8e-3 * (frequency / 150e9) 
    eps = eps_prime + 1j * eps_double_prime
    
    # 3. Get depolarization factor (delta) and shape based on aspect ratio
    delta = compute_depolarization_factor(aspect_ratio)
    shape = 'column' if aspect_ratio > 1.0 else 'plate'

    # 4. Calculate parallel and perpendicular polarizabilities (Eq 8)
    A_par = (eps - 1) / (1 + (eps - 1) * delta)
    A_perp = (eps - 1) / (1 + (eps - 1) * (1 - delta) / 2)
    
    # 5. Calculate Av and Ah based on the shape and the process (Eq 9, 10)
    if shape == 'column':
        if process == 'scattering':
            Av_val = np.abs(A_perp)**2
            Ah_val = (np.abs(A_par)**2 + np.abs(A_perp)**2) / 2
        elif process == 'emission':
            Av_val = np.imag(A_perp)
            Ah_val = (np.imag(A_par) + np.imag(A_perp)) / 2
            
    elif shape == 'plate':
        if process == 'scattering':
            Av_val = np.abs(A_par)**2
            Ah_val = np.abs(A_perp)**2
        elif process == 'emission':
            Av_val = np.imag(A_par)
            Ah_val = np.imag(A_perp)
            
    # 6. Calculate and return gamma (Eq 12)
    gamma = Ah_val / Av_val
    return gamma


def polarization_fraction(frequency, elevation_deg, aspect_ratio=0.5, orientation='horizontal', process='scattering'):
    """
    Computes the polarization fraction (p_gamma) from ice crystals (Eq 11).
    
    Parameters:
    frequency (float): Frequency of the radiation in Hz.
    elevation_deg (float): Telescope elevation angle in degrees.
    aspect_ratio (float): Ratio of symmetry axis to transverse axis (m).
    orientation (str): 'random' or 'horizontal'.
    process (str): 'scattering' or 'emission'.
    
    Returns:
    float: Polarization fraction p_gamma. A negative value indicates horizontal polarization.
    """
    
    # 1. Spherical crystals or random orientations yield no net polarization
    if aspect_ratio == 1.0 or orientation == 'random':
        return 0.0
        
    # 2. Get the gamma parameter representing the particle's polarization response
    gamma = compute_gamma(frequency, aspect_ratio, process)
    
    # 3. Calculate polarization fraction p_gamma based on elevation (Eq 11)
    elev_rad = np.radians(elevation_deg)
    cos2_e = np.cos(elev_rad)**2
    sin2_e = np.sin(elev_rad)**2
    
    numerator = (gamma - 1) * cos2_e
    denominator = gamma * (1 + sin2_e) + cos2_e
    
    p_gamma = - numerator / denominator
    
    return p_gamma








def compute_T_RJ_ice(frequency, altitudes, T_ground, T_ice,  elevation, ice_density, radius_eq, aspect_ratio=1.0, orientation='random', effect='total', polarised=False):
    """
    Computes the Rayleigh-Jeans brightness temperature contribution from ice crystals along a line of sight.
    
    Parameters:
    frequency (array-like): Array of frequencies in Hz. (Nf)
    altitudes (array-like): Array of altitudes in meters. (Nz)
    T_ground (array-like): ground temperature in K. (float))
    T_ice (array-like): Ice crystal temperature in K. (float)
    Pressure (array-like): Array of pressures in hPa. (Nz)
    P_water (array-like): Array of water vapor partial pressures in hPa. (Nz)
    elevation (float): Elevation angle of the telescope in degrees.
    ice_density (array-like): Array of ice water content (in particles/m^3). (Nz, Na)
    radius_eq (array-like): Array of equivalent radii of the ice crystals in meters. (Na)
    aspect_ratio (float): Ratio of symmetry axis to transverse axis (m).
    orientation (str): 'random', 'horizontal', or 'vertical'.
    effect (str): 'total', 'scattering', or 'emission'.
    polarised (bool): Whether you want the intensity (False) or the Q polarized component (True).

    Returns:
    ndarray: The Rayleigh-Jeans brightness temperature array (shape: len(frequency), len(radius_eq)).
    """

    # 1. Geometry and airmass
    zenith_angle = 90.0 - elevation
    m = 1.0 / (np.cos(np.radians(zenith_angle)) + 0.50572 * (96.07995 - zenith_angle)**(-1.6364))
    dz = np.diff(altitudes) # Shape: (Nz-1,)

    # Particle volumes
    V = (4.0 / 3.0) * np.pi * radius_eq**3 # Shape: (Na,)
    
    # Complex polarizability
    A = compute_polarizability(frequency, aspect_ratio, orientation) # Shape: (Nf,)

    # Cross-sections
    sigma_sca = sigma_scattering(frequency[:, None], V[None, :], A[:, None]) # Shape: (Nf, Na)
    sigma_abs = sigma_absorption(frequency[:, None], V[None, :], A[:, None]) # Shape: (Nf, Na)

    # Ice attenuation coefficients (Alpha = density * sigma)
    alpha_sca = ice_density[:, None, :] * sigma_sca[None, :, :] # Shape: (Nz, Nf, Na)
    alpha_abs = ice_density[:, None, :] * sigma_abs[None, :, :] # Shape: (Nz, Nf, Na)

    # Midpoint averages for each layer
    alpha_sca_mid = (alpha_sca[:-1, :, :] + alpha_sca[1:, :, :]) / 2.0
    alpha_abs_mid = (alpha_abs[:-1, :, :] + alpha_abs[1:, :, :]) / 2.0

    # Optical thicknesses (d_tau) for each process
    d_tau_sca = alpha_sca_mid * dz[:, None, None] # Shape: (Nz-1, Nf, Na)
    d_tau_abs = alpha_abs_mid * dz[:, None, None] # Shape: (Nz-1, Nf, Na)
    


    # 3. Initialization of components
    T_layers_sca = 0.0
    T_layers_abs = 0.0

    # --- Calculation of the Scattering contribution ---
    if effect in ['scattering', 'total']:
        # Eq 3: T_source is T_ground / 2
        T_source_sca = T_ground / 2.0 
        
        if polarised:
            p_gamma_sca = polarization_fraction(frequency, elevation, aspect_ratio, orientation, process='scattering')
            T_source_sca = T_source_sca * p_gamma_sca[:, None] # Apply polarization
            
        T_layers_sca = T_source_sca * (1 - np.exp(-d_tau_sca * m))

    # --- Calculation of the Emission contribution ---
    if effect in ['emission', 'total']:
        # Eq 4: T_source is the local thermodynamic temperature of the layer
        T_source_abs = T_ice
        
        if polarised:
            p_gamma_abs = polarization_fraction(frequency, elevation, aspect_ratio, orientation, process='emission')
            T_source_abs = T_source_abs * p_gamma_abs[None, :, None] # Apply polarization
            
        T_layers_abs = T_source_abs * (1 - np.exp(-d_tau_abs * m))

    # 4. Selection of the desired effect
    if effect == 'scattering':
        T_layers = T_layers_sca
    elif effect == 'emission':
        T_layers = T_layers_abs
    else: # 'total'
        T_layers = T_layers_sca + T_layers_abs

    # 5. Integration over all altitudes (Sum along the line of sight)
    T_sky = np.sum(T_layers, axis=0) # Final shape: (Nf, Na)
    
    return T_sky

In [41]:
                
frequency = np.array([90e9]) # Frequency in Hz (90 GHz)                  
T_ground = 270.0        
T_sky_zenith = 10.0 
T_sky_ice = 250 # K    

elevation = 45.0 # degrees 
aspect_ratio = 0.1      # Plates
D = 100e-6  

r_eq = np.array([D/2]) # Equivalent radius of the ice crystals in meters (Na)
dz = 100.0                             
altitude_layers = np.arange(6000, 8000, dz) 
n_z = 1e5     
#let's create the constant ice layer
ice_denisity = n_z * np.ones_like(altitude_layers) # particles/m^3
#Lets's make ice density in shape Nz, Na
ice_density = ice_denisity[:, None] * np.ones((len(altitude_layers), len(r_eq))) # Shape: (Nz, Na)

In [42]:
#Let's calculate the brightness temperature contribution from the ice layer
T_sky_ice_contribution = compute_T_RJ_ice(frequency, altitude_layers, T_ground, T_sky_ice, elevation, ice_density, r_eq, aspect_ratio=aspect_ratio, orientation='horizontal', effect='scattering', polarised=True)

In [43]:
print(T_sky_ice_contribution)

[[-0.00623519]]


In [9]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import constants

pi = constants.pi 
c = constants.c


def create_constant_ice_layer(altitudes, layer_bottom, layer_top, ice_density):
    """
    Create a constant ice layer with specified density between given altitudes.

    Parameters:
    altitudes (numpy array): Array of altitude values in meters.
    layer_bottom (float): Bottom altitude of the ice layer in meters.
    layer_top (float): Top altitude of the ice layer in meters.
    ice_density (float): Density of ice particles in particles/m^3.

    Returns:
    numpy array: Array of particle densities corresponding to the input altitudes.
    """
    n = np.zeros_like(altitudes)  # Initialize an array of zeros
    in_layer = (altitudes >= layer_bottom) & (altitudes <= layer_top)  # Find indices within the layer
    n[in_layer] = ice_density  # Set density for those indices
    return n

def sigma_scattering(frequency, volume, A):
    """
    Calculate the scattering cross section for ice crystals.
    
    #Parameters:
    frequency (float): The frequency of the incident light (in Hz).
    volume (float): The volume of the ice crystal particle (in m^3).
    A (float): A constant that depends on the properties of the ice crystal (dielectric properties, shape, polarizability).
    
    Returns:
    float: The scattering cross section (in m^2).
    """
    w = 2 * pi * frequency #angular frequency

    sigma_sca = w**4*volume**2*np.abs(A)**2/(6*pi*c**4)
    
    return sigma_sca

def sigma_absorption(frequency, volume, A):
    """
    Calculate the absorption cross section for ice crystals.
    
    #Parameters:
    frequency (float): The frequency of the incident light (in Hz).
    volume (float): The volume of the ice crystal particle (in m^3).
    A (float): A constant that depends on the properties of the ice crystal (dielectric properties, shape, polarizability).
    
    Returns:
    float: The absorption cross section (in m^2).
    """
    w = 2 * pi * frequency #angular frequency

    sigma_abs = w*volume*np.imag(A)/c

    return sigma_abs

def compute_depolarization_factor(aspect_ratio):
    """
    Calculates the depolarization factor (Delta) along the symmetry axis
    for a spheroid given its aspect ratio.
    
    Parameters:
    aspect_ratio (float): Ratio of symmetry axis to transverse axis (m).
                          m = 1.0 (sphere), m > 1.0 (column), m < 1.0 (plate).
    
    Returns:
    float: Depolarization factor Delta.
    """
    m = aspect_ratio
    
    if m == 1.0:
        # Sphere
        return 1.0 / 3.0
        
    elif m > 1.0:
        # Prolate spheroid (Column)
        e = np.sqrt(1.0 - (1.0 / m)**2)
        delta = ((1.0 - e**2) / e**2) * ((1.0 / (2.0 * e)) * np.log((1.0 + e) / (1.0 - e)) - 1.0)
        return delta
        
    else:
        # Oblate spheroid (Plate)
        e = np.sqrt(1.0 - m**2)
        delta = (1.0 / e**2) * (1.0 - (np.sqrt(1.0 - e**2) / e) * np.arcsin(e))
        return delta

def compute_intrinsic_polarizabilities(frequency, aspect_ratio):
    """Computes the inherent polarizabilities (A_par, A_perp) relative to the particle's symmetry axis."""
    eps_prime = 3.16
    eps_double_prime = 8e-3 * (frequency / 150e9) 
    eps = eps_prime + 1j * eps_double_prime
    
    if aspect_ratio == 1.0:
        A_par = 3 * (eps - 1) / (eps + 2)
        return A_par, A_par
    
    delta = compute_depolarization_factor(aspect_ratio)
    A_par = (eps - 1) / (1 + (eps - 1) * delta)
    A_perp = (eps - 1) / (1 + (eps - 1) * (1 - delta) / 2.0)
    
    return A_par, A_perp


def compute_effective_polarizability(frequency, aspect_ratio, elevation, stokes_param='I'):
    """
    Projects the polarizabilities onto the telescope based on aerodynamics and elevation.
    Correctly handles the incoherent azimuthal averaging for spinning columns.
    """
    # 1. Get intrinsic complex polarizabilities
    A_par, A_perp = compute_intrinsic_polarizabilities(frequency, aspect_ratio)
    
    # 2. Pre-calculate the physical intensities (abs^2) and absorption (imag)
    abs2_par = np.abs(A_par)**2
    abs2_perp = np.abs(A_perp)**2
    
    imag_par = np.imag(A_par)
    imag_perp = np.imag(A_perp)
    
    # 3. Map to Earth-frame based on aerodynamics (Incoherent Averaging)
    if aspect_ratio == 1.0:
        # Sphere
        abs2_v, abs2_h = abs2_par, abs2_par
        imag_v, imag_h = imag_par, imag_par
        
    elif aspect_ratio < 1.0:
        # Plates (Fall flat, symmetry axis is vertical)
        abs2_v, abs2_h = abs2_par, abs2_perp
        imag_v, imag_h = imag_par, imag_perp
        
    else:
        # Columns (Fall flat, spin randomly in azimuth)
        # Vertical sees pure transverse. Horizontal is the average of parallel and transverse.
        abs2_v = abs2_perp
        abs2_h = (abs2_par + abs2_perp) / 2.0
        
        imag_v = imag_perp
        imag_h = (imag_par + imag_perp) / 2.0
        
    # 4. Project onto Telescope Sensors based on Stokes parameter
    eps_rad = np.radians(elevation)
    
    if stokes_param == 'I':
        eff_abs2 = 0.5 * (abs2_v * np.cos(eps_rad)**2 + abs2_h * (1.0 + np.sin(eps_rad)**2))
        eff_imag = 0.5 * (imag_v * np.cos(eps_rad)**2 + imag_h * (1.0 + np.sin(eps_rad)**2))
        
    elif stokes_param == 'Q':
        eff_abs2 = 0.5 * (abs2_v - abs2_h) * np.cos(eps_rad)**2
        eff_imag = 0.5 * (imag_v - imag_h) * np.cos(eps_rad)**2
        
    else:
        raise ValueError("stokes_param must be 'I' or 'Q'")

    return eff_abs2, eff_imag

In [10]:
import numpy as np

def polarization_fraction_2(frequency, elevation_deg, aspect_ratio=0.5, process='scattering'):
    """
    Computes the polarization fraction (Q/I) directly from the effective polarizabilities,
    bypassing the intermediate gamma ratio calculation.
    
    Parameters:
    frequency (float or array): Frequency in Hz.
    elevation_deg (float or array): Telescope elevation angle in degrees.
    aspect_ratio (float): Ratio of symmetry axis to transverse axis. Defaults to 0.5 (plate).
    process (str): 'scattering' or 'emission'.
    
    Returns:
    float or array: The polarization fraction p_gamma.
    """
    
    # 1. Get the effective polarizabilities for Total Intensity (I)
    eff_abs2_I, eff_imag_I = compute_effective_polarizability(
        frequency, aspect_ratio, elevation_deg, stokes_param='I'
    )
    
    # 2. Get the effective polarizabilities for Linear Polarization (Q)
    eff_abs2_Q, eff_imag_Q = compute_effective_polarizability(
        frequency, aspect_ratio, elevation_deg, stokes_param='Q'
    )
    
    # 3. Calculate the fraction depending on the physical process
    if process == 'scattering':
        # Scattering scales with the absolute square of the polarizability
        p_frac = eff_abs2_Q / eff_abs2_I
        
    elif process == 'emission':
        # Thermal emission scales with the imaginary part of the polarizability (absorption)
        p_frac = eff_imag_Q / eff_imag_I
        
    else:
        raise ValueError("Process must be 'scattering' or 'emission'.")
        
    return p_frac

In [11]:
#Let's compare the two functions polarization_fraction and polarization_fraction_2 for a range of aspect ratios and elevations
frequencies = np.array([90e9]) # 90 GHz
elevations = np.array([10, 30, 45, 60, 80]) # degrees
aspect_ratios = np.array([0.1, 0.5, 1.0, 2.0, 5.0]) # Plates to columns
results_1 = np.zeros((len(aspect_ratios), len(elevations)))
results_2 = np.zeros((len(aspect_ratios), len(elevations)))
for i, ar in enumerate(aspect_ratios):
    for j, elev in enumerate(elevations):
        results_1[i, j] = polarization_fraction(frequencies[0], elev, aspect_ratio=ar, orientation='horizontal', process='scattering')
        results_2[i, j] = polarization_fraction_2(frequencies[0], elev, aspect_ratio=ar, process='scattering')
print("Polarization fractions from polarization_fraction (Eq 11):")
print(results_1)
print("\nPolarization fractions from polarization_fraction_2 (direct from polarizabilities):")
print(results_2)    

Polarization fractions from polarization_fraction (Eq 11):
[[-0.68474412 -0.45837653 -0.26508191 -0.11702972 -0.01279838]
 [-0.32101699 -0.23140902 -0.14322484 -0.06682679 -0.00761291]
 [ 0.          0.          0.          0.          0.        ]
 [-0.17631571 -0.13110812 -0.08374551 -0.04018989 -0.00468202]
 [-0.34526154 -0.24761746 -0.15249176 -0.0708443  -0.00804375]]

Polarization fractions from polarization_fraction_2 (direct from polarizabilities):
[[-0.68474412 -0.45837653 -0.26508191 -0.11702972 -0.01279838]
 [-0.32101699 -0.23140902 -0.14322484 -0.06682679 -0.00761291]
 [ 0.          0.          0.          0.          0.        ]
 [-0.17631571 -0.13110812 -0.08374551 -0.04018989 -0.00468202]
 [-0.34526154 -0.24761746 -0.15249176 -0.0708443  -0.00804375]]


In [12]:
import numpy as np

def compute_T_RJ_ice2(frequency, altitudes, T_ground, T_ice, elevation, ice_density, radius_eq, aspect_ratio=1.0, process='total', stokes_param='I'):
    """
    Computes the Rayleigh-Jeans brightness temperature contribution from ice crystals 
    along a line of sight for a specific Stokes parameter (I or Q).
    
    Parameters:
    frequency (array-like): Array of frequencies in Hz. (Nf)
    altitudes (array-like): Array of altitudes in meters. (Nz)
    T_ground (float or array): Ground temperature in K.
    T_ice (float or array): Ice crystal temperature in K. (Nz or scalar)
    elevation (float): Elevation angle of the telescope in degrees.
    ice_density (array-like): Array of ice water content (in particles/m^3). (Nz, Na)
    radius_eq (array-like): Array of equivalent radii of the ice crystals in meters. (Na)
    aspect_ratio (float): Ratio of symmetry axis to transverse axis.
    process (str): 'scattering', 'emission', or 'total'.
    stokes_param (str): 'I' (Total Intensity) or 'Q' (Linear Polarization).

    Returns:
    ndarray: The Rayleigh-Jeans brightness temperature array (shape: Nf, Na).
    """
    c = constants.c
    
    # 1. Geometry and airmass
    zenith_angle = 90.0 - elevation
    m = 1.0 / (np.cos(np.radians(zenith_angle)) + 0.50572 * (96.07995 - zenith_angle)**(-1.6364))
    dz = np.diff(altitudes) # Shape: (Nz-1,)

    # Particle volumes
    V = (4.0 / 3.0) * np.pi * radius_eq**3 # Shape: (Na,)
    
    # 2. Get the EXACT Effective Cross-Section Multipliers based on Stokes parameter
    # This automatically handles the aerodynamic alignment and telescope projection!
    eff_abs2_sca, eff_imag_abs = compute_effective_polarizability(frequency, aspect_ratio, elevation, stokes_param=stokes_param)

    # 3. Calculate Cross-Sections Directly
    # Shape of resulting sigmas: (Nf, Na)
    sigma_sca = (1.0 / (6.0 * np.pi)) * ((2 * np.pi * frequency[:, None] / c)**4) * (V[None, :]**2) * eff_abs2_sca[:, None]
    sigma_abs = (2 * np.pi * frequency[:, None] / c) * V[None, :] * eff_imag_abs[:, None]

    # Ice attenuation coefficients (Alpha = density * sigma)
    alpha_sca = ice_density[:, None, :] * sigma_sca[None, :, :] # Shape: (Nz, Nf, Na)
    alpha_abs = ice_density[:, None, :] * sigma_abs[None, :, :] # Shape: (Nz, Nf, Na)

    # Midpoint averages for each layer
    alpha_sca_mid = (alpha_sca[:-1, :, :] + alpha_sca[1:, :, :]) / 2.0
    alpha_abs_mid = (alpha_abs[:-1, :, :] + alpha_abs[1:, :, :]) / 2.0

    # Optical thicknesses (d_tau) for each process
    d_tau_sca = alpha_sca_mid * dz[:, None, None] # Shape: (Nz-1, Nf, Na)
    d_tau_abs = alpha_abs_mid * dz[:, None, None] # Shape: (Nz-1, Nf, Na)
    
    # Ensure T_ice matches layer dimensions if it's provided as an array
    if isinstance(T_ice, np.ndarray) and T_ice.size == len(altitudes):
        T_ice_mid = (T_ice[:-1] + T_ice[1:]) / 2.0
        T_ice_mid = T_ice_mid[:, None, None]
    else:
        T_ice_mid = T_ice

    # 4. Initialization of components
    T_layers_sca = 0.0
    T_layers_abs = 0.0

    # --- Calculation of the Scattering contribution ---
    if process in ['scattering', 'total']:
        # The source of scattering is the thermal ground emission
        T_source_sca = T_ground / 2.0 
        
        # d_tau_sca natively contains the exact I or Q projection, 
        # so this automatically outputs the unpolarized T_I or the polarized T_Q!
        T_layers_sca = T_source_sca * (1 - np.exp(-d_tau_sca * m))

    # --- Calculation of the Emission contribution ---
    if process in ['emission', 'total']:
        # The source of emission is the local thermodynamic temperature
        T_source_abs = T_ice_mid
        
        # d_tau_abs natively contains the exact I or Q projection
        T_layers_abs = T_source_abs * (1 - np.exp(-d_tau_abs * m))

    # 5. Selection of the desired effect
    if process == 'scattering':
        T_layers = T_layers_sca
    elif process == 'emission':
        T_layers = T_layers_abs
    else: # 'total'
        T_layers = T_layers_sca + T_layers_abs

    # 6. Integration over all altitudes (Sum along the line of sight)
    T_sky = np.sum(T_layers, axis=0) # Final shape: (Nf, Na)
    
    return T_sky

In [38]:
frequency = np.array([90e9]) # Frequency in Hz (90 GHz)                  
T_ground = 270.0        
T_sky_zenith = 10.0 
T_sky_ice = 250 # K    

elevation = 45.0 # degrees 
aspect_ratio = 0.1      # Plates
D = 100e-6  

r_eq = np.array([D/2]) # Equivalent radius of the ice crystals in meters (Na)
dz = 100.0                             
altitude_layers = np.arange(6000, 8000, dz) 
n_z = 1e5     
#let's create the constant ice layer
ice_denisity = n_z * np.ones_like(altitude_layers) # particles/m^3
#Lets's make ice density in shape Nz, Na
ice_density = ice_denisity[:, None] * np.ones((len(altitude_layers), len(r_eq))) # Shape: (Nz, Na)

In [39]:
#Let's calculate the brightness temperature contribution from the ice layer
T_sky_ice_contribution2 = compute_T_RJ_ice2(frequency, altitude_layers, T_ground, T_sky_ice, elevation, ice_density, r_eq, aspect_ratio=0.1, process='scattering', stokes_param='Q')

In [40]:
print("Brightness temperature contribution to Stokes Q from the ice layer:")
print(T_sky_ice_contribution2)

Brightness temperature contribution to Stokes Q from the ice layer:
[[-0.00492871]]


In [44]:
import numpy as np
from scipy import constants

def compute_T_RJ_ice2(frequency, altitudes, T_ground, T_ice, elevation, ice_density, radius_eq, aspect_ratio=1.0, process='total', stokes_param='I'):
    """
    Computes the Rayleigh-Jeans brightness temperature contribution from ice crystals 
    along a line of sight for a specific Stokes parameter (I or Q).
    
    Parameters:
    frequency (array-like): Array of frequencies in Hz. (Nf)
    altitudes (array-like): Array of altitudes in meters. (Nz)
    T_ground (float or array): Ground temperature in K.
    T_ice (float or array): Ice crystal temperature in K. (Nz or scalar)
    elevation (float): Elevation angle of the telescope in degrees.
    ice_density (array-like): Array of ice water content (in particles/m^3). (Nz, Na)
    radius_eq (array-like): Array of equivalent radii of the ice crystals in meters. (Na)
    aspect_ratio (float or array): Ratio of symmetry axis to transverse axis.
    process (str): 'scattering', 'emission', or 'total'.
    stokes_param (str): 'I' (Total Intensity) or 'Q' (Linear Polarization).

    Returns:
    ndarray: The Rayleigh-Jeans brightness temperature array (shape: Nf, Na).
    """
    c = constants.c
    
    # 1. Geometry and airmass
    zenith_angle = 90.0 - elevation
    # Check for zenith to avoid division by zero edge cases, use a standard airmass model
    m = 1.0 / (np.cos(np.radians(zenith_angle)) + 0.50572 * (96.07995 - zenith_angle)**(-1.6364))
    dz = np.diff(altitudes) # Shape: (Nz-1,)

    # Particle volumes
    V = (4.0 / 3.0) * np.pi * radius_eq**3 # Shape: (Na,)
    
    # 2. Get Effective Cross-Section Multipliers
    # We MUST fetch both the total intensity (I) AND the requested Stokes parameter
    eff_abs2_sca_req, eff_imag_abs_req = compute_effective_polarizability(frequency, aspect_ratio, elevation, stokes_param=stokes_param)
    eff_abs2_sca_I, eff_imag_abs_I = compute_effective_polarizability(frequency, aspect_ratio, elevation, stokes_param='I')

    # 3. Calculate Cross-Sections Directly
    k_4 = (2 * np.pi * frequency[:, None] / c)**4
    k_1 = (2 * np.pi * frequency[:, None] / c)
    geom_factor = 1.0 / (6.0 * np.pi)

    # Scattering cross-sections: Shape (Nf, Na)
    sigma_sca_req = geom_factor * k_4 * (V[None, :]**2) * eff_abs2_sca_req[:, None]
    sigma_sca_I   = geom_factor * k_4 * (V[None, :]**2) * eff_abs2_sca_I[:, None]

    # Absorption cross-sections: Shape (Nf, Na)
    sigma_abs_req = k_1 * V[None, :] * eff_imag_abs_req[:, None]
    sigma_abs_I   = k_1 * V[None, :] * eff_imag_abs_I[:, None]

    # Ice attenuation coefficients (Alpha = density * sigma): Shape (Nz, Nf, Na)
    alpha_sca_req = ice_density[:, None, :] * sigma_sca_req[None, :, :] 
    alpha_sca_I   = ice_density[:, None, :] * sigma_sca_I[None, :, :] 
    
    alpha_abs_req = ice_density[:, None, :] * sigma_abs_req[None, :, :] 
    alpha_abs_I   = ice_density[:, None, :] * sigma_abs_I[None, :, :] 

    # Midpoint averages for each layer
    alpha_sca_req_mid = (alpha_sca_req[:-1, :, :] + alpha_sca_req[1:, :, :]) / 2.0
    alpha_sca_I_mid   = (alpha_sca_I[:-1, :, :] + alpha_sca_I[1:, :, :]) / 2.0
    
    alpha_abs_req_mid = (alpha_abs_req[:-1, :, :] + alpha_abs_req[1:, :, :]) / 2.0
    alpha_abs_I_mid   = (alpha_abs_I[:-1, :, :] + alpha_abs_I[1:, :, :]) / 2.0

    # Optical thicknesses (d_tau) for each process: Shape (Nz-1, Nf, Na)
    d_tau_sca_req = alpha_sca_req_mid * dz[:, None, None]
    d_tau_sca_I   = alpha_sca_I_mid * dz[:, None, None]
    
    d_tau_abs_req = alpha_abs_req_mid * dz[:, None, None]
    d_tau_abs_I   = alpha_abs_I_mid * dz[:, None, None]
    
    # Ensure T_ice matches layer dimensions if it's provided as an array
    if isinstance(T_ice, np.ndarray) and T_ice.size == len(altitudes):
        T_ice_mid = (T_ice[:-1] + T_ice[1:]) / 2.0
        T_ice_mid = T_ice_mid[:, None, None]
    else:
        T_ice_mid = T_ice

    # 4. Radiative Transfer (Layer by Layer Formulation)
    T_layers_sca = 0.0
    T_layers_abs = 0.0

    # --- Calculation of the Scattering contribution ---
    if process in ['scattering', 'total']:
        # Ground source function (half-hemisphere integration)
        T_source_sca = T_ground / 2.0 
        
        # Calculate the macroscopic fraction p_gamma for this specific layer
        # np.divide avoids ZeroDivisionErrors if d_tau_I is 0 (perfectly clear air)
        p_gamma_sca = np.divide(d_tau_sca_req, d_tau_sca_I, out=np.zeros_like(d_tau_sca_I), where=(d_tau_sca_I != 0))
        
        # Radiative Transfer for Total Intensity (I)
        T_layers_sca_I = T_source_sca * (1 - np.exp(-d_tau_sca_I * m))
        
        # Final output for this layer: fraction * Intensity
        T_layers_sca = p_gamma_sca * T_layers_sca_I

    # --- Calculation of the Emission contribution ---
    if process in ['emission', 'total']:
        T_source_abs = T_ice_mid
        
        # Calculate the macroscopic fraction p_gamma for this specific layer
        p_gamma_abs = np.divide(d_tau_abs_req, d_tau_abs_I, out=np.zeros_like(d_tau_abs_I), where=(d_tau_abs_I != 0))
        
        # Radiative Transfer for Total Intensity (I)
        T_layers_abs_I = T_source_abs * (1 - np.exp(-d_tau_abs_I * m))
        
        # Final output for this layer: fraction * Intensity
        T_layers_abs = p_gamma_abs * T_layers_abs_I

    # 5. Selection of the desired effect
    if process == 'scattering':
        T_layers = T_layers_sca
    elif process == 'emission':
        T_layers = T_layers_abs
    else: # 'total'
        T_layers = T_layers_sca + T_layers_abs

    # 6. Integration over all altitudes (Sum along the line of sight)
    T_sky = np.sum(T_layers, axis=0) # Final shape: (Nf, Na)
    
    return T_sky

In [45]:
frequency = np.array([90e9]) # Frequency in Hz (90 GHz)                  
T_ground = 270.0        
T_sky_zenith = 10.0 
T_sky_ice = 250 # K    

elevation = 45.0 # degrees 
aspect_ratio = 0.1      # Plates
D = 100e-6  

r_eq = np.array([D/2]) # Equivalent radius of the ice crystals in meters (Na)
dz = 100.0                             
altitude_layers = np.arange(6000, 8000, dz) 
n_z = 1e5     
#let's create the constant ice layer
ice_denisity = n_z * np.ones_like(altitude_layers) # particles/m^3
#Lets's make ice density in shape Nz, Na
ice_density = ice_denisity[:, None] * np.ones((len(altitude_layers), len(r_eq))) # Shape: (Nz, Na)

#Let's calculate the brightness temperature contribution from the ice layer
T_sky_ice_contribution2 = compute_T_RJ_ice2(frequency, altitude_layers, T_ground, T_sky_ice, elevation, ice_density, r_eq, aspect_ratio=0.1, process='scattering', stokes_param='Q')

In [46]:
print("Brightness temperature contribution to Stokes Q from the ice layer:")
print(T_sky_ice_contribution2)

Brightness temperature contribution to Stokes Q from the ice layer:
[[-0.00492869]]
